In [1]:
# Check last executions
past_time = "36d"
notebook_client_name = "usdf_efd"

# N days for historical evaluation
ndays = 5

# M1M3 Bump Test Log Error Analysis and Measured Forces

## Overview

This notebook is designed to facilitate the analysis of M1M3 force actuator “Bump Test” logs and the visualization of individual actuator performance.
It combines:

- **Bump Test Log Analysis:** Queries and processes recent bump test logs from the EFD to identify failed force actuators and extract relevant error information.

- **Script Execution Details:** Retrieves and displays execution logs for the `check_actuators.py` script, showing the duration, start/end times, and final statuses of each run.

- **Per-Actuator Visualization:** Plots measured and commanded forces, following errors, and other key metrics for individual actuators, helping users diagnose and understand any anomalies or deviations from expected values.

This integrated approach allows for efficient troubleshooting and performance assessment of M1M3 actuators based on either real-time or historical data.

## Setup and Imports

In [2]:
import re
from astropy.time import Time, TimeDelta
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import asyncio

# Matplotlib imports (still used by your existing plotting code)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import matplotlib.dates as mdates
from matplotlib.ticker import FormatStrFormatter

# EFD imports
from lsst_efd_client import EfdClient

# M1M3/FA definitions
from lsst.ts.xml.tables.m1m3 import FATable, force_actuator_from_id
from lsst.ts.xml.enums.MTM1M3 import BumpTest

from IPython.display import display, HTML

# Bokeh imports (for the new UI)
from bokeh.io import output_notebook, show, curdoc
from bokeh.models import (
    Select,
    CheckboxGroup,
    Button,
    ColumnDataSource,
    DataTable,
    TableColumn,
    Div,
    CustomJS,
)
from bokeh.layouts import column, row
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.layouts import gridplot

# If running in a Jupyter Notebook, activate Bokeh inline plotting
output_notebook()

Loading BokehJS ...

In [3]:
def convert_to_hours(past_time):
    """
    Convert a string that can be either a number of hours (e.g. '6h')
    or a number of days (e.g. '3d') to total number of hours (int).
    """
    match = re.match(r"(\d+)([dh]?)", past_time)
    if match:
        value, unit = match.groups()
        value = int(value)
        if unit == "d":
            return value * 24
        else:
            return value
    else:
        raise ValueError(
            "Invalid time format. Please use a format "
            "like '6h' for hours or '3d' for days."
        )

### Query and Filter Script Queue Logs

In [4]:
async def query_script_queue_logs(
    start_time_str: str, end_time_str: str, client_name="usdf_efd"
):
    """
    Queries the log messages related to the script queue from the EFD.

    Parameters
    ----------
    start_time_str : str
        Start time in ISO format, e.g. "2024-11-04T12:00:00".
    end_time_str : str
        End time in ISO format, e.g. "2024-11-04T13:00:00".
    client_name : str, optional
        Name of the EFD client. Defaults to "usdf_efd".

    Returns
    -------
    pd.DataFrame
        DataFrame with script queue logs in the given time window.
    """
    # Convert string times to astropy Time
    start = Time(start_time_str, format="isot", scale="utc")
    end = Time(end_time_str, format="isot", scale="utc")

    # Create the EFD client
    possible_clients = ["summit_efd", "usdf_efd"]
    if client_name not in possible_clients:
        print(f"Invalid client name. Possible clients: {possible_clients}")
        return None

    client = EfdClient(client_name)
    script_logs = await client.select_time_series(
        topic_name="lsst.sal.ScriptQueue.logevent_script",
        fields="*",
        start=start,
        end=end,
    )

    return script_logs


def filter_and_process_queue_logs(script_logs):
    """
    Filters and processes the script logs DataFrame for a specific script (maintel/m1m3/check_actuators.py).

    Parameters
    ----------
    script_logs : pd.DataFrame
        DataFrame containing the script logs.

    Returns
    -------
    pd.DataFrame
        Processed DataFrame containing only relevant logs for the check_actuators script.
    """
    processState_mapping = {
        0: "UNKNOWN",
        1: "LOADING",
        2: "CONFIGURED",
        3: "RUNNING",
        4: "DONE",
        5: "LOADFAILED",
        6: "CONFIGURE_FAILED",
        7: "TERMINATED",
    }

    scriptState_mapping = {
        0: "UNKNOWN",
        1: "UNCONFIGURED",
        2: "CONFIGURED",
        3: "RUNNING",
        4: "PAUSED",
        5: "ENDING",
        6: "STOPPING",
        7: "FAILING",
        8: "DONE",
        9: "STOPPED",
        10: "FAILED",
        11: "CONFIGURE_FAILED",
    }

    # Script path and salIndex we’re focusing on
    path = "maintel/m1m3/check_actuators.py"
    salIndex = 1

    df_script_logs = pd.DataFrame(script_logs)

    # Filter logs for the specific script path and salIndex
    df_filtered_logs = df_script_logs[
        (df_script_logs["path"] == path) & (df_script_logs["salIndex"] == salIndex)
    ]

    # Reindex by private_rcvStamp, convert to UTC
    df_filtered_logs = df_filtered_logs.set_index("private_rcvStamp")
    df_filtered_logs.index = pd.to_datetime(df_filtered_logs.index, unit="s")
    df_filtered_logs.index = df_filtered_logs.index - timedelta(seconds=37)

    # Convert all columns starting with 'timestamp' to datetime, also shift TAI->UTC
    timestamp_columns = [
        col for col in df_filtered_logs.columns if col.startswith("timestamp")
    ]
    for col in timestamp_columns:
        df_filtered_logs[col] = pd.to_datetime(df_filtered_logs[col], unit="s")
        df_filtered_logs[col] = df_filtered_logs[col] - timedelta(seconds=37)

    # Map numeric process/script states to strings
    df_filtered_logs["processState_str"] = df_filtered_logs["processState"].map(
        processState_mapping
    )
    df_filtered_logs["scriptState_str"] = df_filtered_logs["scriptState"].map(
        scriptState_mapping
    )

    # Remove unneeded columns
    columns_to_remove = [
        "blockId",
        "blockSize",
        "cmdId",
        "executionId",
        "private_efdStamp",
        "private_kafkaStamp",
        "private_origin",
        "private_revCode",
        "private_sndStamp",
        "private_seqNum",
        "scriptBlockIndex",
        "isStandard",
        "private_identity",
        "processState",
        "scriptState",
    ]
    df_filtered_logs.drop(columns=columns_to_remove, inplace=True, errors="ignore")

    # Sort from most recent to oldest
    df_filtered_logs.sort_index(ascending=False, inplace=True)

    return df_filtered_logs


def extract_execution_details(df_check_actuators_log):
    """
    Extracts execution details from the script logs DataFrame.

    Parameters
    ----------
    df_check_actuators_log : pd.DataFrame
        DataFrame containing the filtered script logs.

    Returns
    -------
    pd.DataFrame
        DataFrame containing execution times, durations, and final statuses.
    """
    # Initialize an empty list to store execution details
    execution_data = []

    # Loop over each unique scriptSalIndex
    for script_sal_index in df_check_actuators_log["scriptSalIndex"].unique():
        df_sal = df_check_actuators_log[
            df_check_actuators_log["scriptSalIndex"] == script_sal_index
        ]
        df_sal = df_sal.sort_index()  # Ensure chronological order

        # Calculate start and end times for each execution
        start_time = df_sal["timestampProcessStart"].min()
        end_time = df_sal["timestampProcessEnd"].max()

        # Get the final process and script status
        final_process_status = df_sal["processState_str"].iloc[-1]
        final_script_status = df_sal["scriptState_str"].iloc[-1]

        execution_data.append(
            {
                "scriptSalIndex": script_sal_index,
                "start_time": start_time,
                "end_time": end_time,
                "FinalProcessStatus": final_process_status,
                "FinalScriptStatus": final_script_status,
            }
        )

    # Create the DataFrame
    df_executions = pd.DataFrame(execution_data)

    if not df_executions.empty:
        # Calculate duration in minutes
        df_executions["duration_minutes"] = (
            df_executions["end_time"] - df_executions["start_time"]
        ).dt.total_seconds() / 60.0

        # Format durations to .2f
        df_executions["duration_minutes"] = df_executions["duration_minutes"].apply(
            lambda x: "{:.2f}".format(x)
        )

        # Reorder columns
        cols = [
            "scriptSalIndex",
            "start_time",
            "end_time",
            "duration_minutes",
            "FinalProcessStatus",
            "FinalScriptStatus",
        ]
        df_executions = df_executions[cols]

        # Sort by start time in descending order
        df_executions = df_executions.sort_values("start_time", ascending=False)
    else:
        print("No executions found.")

    return df_executions


## Bump Test Queries and Processing

### Query Bump Logs

In [5]:
async def query_bump_logs(start_date: str, end_date: str, client_name="summit_efd"):
    """
    Queries the log messages related to bump tests from the EFD.

    Parameters
    ----------
    start_date : str
        Start date of the query in ISO format, e.g. "2024-11-04T12:00:00".
    end_date : str
        End date of the query in ISO format, e.g. "2024-11-04T13:00:00".
    client_name : str, optional
        Name of the EFD client. Defaults to "summit_efd".

    Returns
    -------
    pd.DataFrame
        DataFrame of bump log messages with the requested fields.
    """
    # Convert strings to datetime, then to astropy Time
    start_dt = datetime.fromisoformat(start_date)
    end_dt = datetime.fromisoformat(end_date)

    possible_clients = ["summit_efd", "usdf_efd"]
    if client_name not in possible_clients:
        print(f"Invalid client name. Possible clients: {possible_clients}")
        return None

    client = EfdClient(client_name)

    try:
        bump_logs = await client.select_time_series(
            topic_name="lsst.sal.MTM1M3.logevent_logMessage",
            fields=["message"],
            start=Time(start_dt.isoformat(), format="isot", scale="utc"),
            end=Time(end_dt.isoformat(), format="isot", scale="utc"),
        )
        return bump_logs
    except Exception as e:
        print(
            f"Error querying data from {start_dt.isoformat()} to {end_dt.isoformat()}: {e}"
        )
        return pd.DataFrame()


def process_bump_logs(bump_logs, expected_force_range=222, tolerance=5):
    """
    Processes bump log messages to extract relevant information and calculate deviations
    from expected forces.

    Parameters
    ----------
    bump_logs : pd.DataFrame
        DataFrame containing bump log messages.
    expected_force_range : float, optional
        Expected magnitude of the applied force.
    tolerance : float, optional
        Tolerance for the allowed variation in force.

    Returns
    -------
    pd.DataFrame
        Processed DataFrame with extracted and calculated data.
    """
    df_filtered_bump_log = bump_logs[
        bump_logs["message"].str.contains("Failed FA")
    ].copy()

    if df_filtered_bump_log.empty:
        print("No failed FA messages found in bump logs.")
        return pd.DataFrame()

    # Extract relevant info from the message
    df_filtered_bump_log.loc[:, "ID"] = df_filtered_bump_log["message"].str.extract(
        r"FA ID (\d+)"
    )
    orientation_index = df_filtered_bump_log["message"].str.extract(r"\((X|Y|Z)(\d+)\)")
    df_filtered_bump_log.loc[:, "Orientation"] = orientation_index[0]
    df_filtered_bump_log.loc[:, "Index"] = orientation_index[1]
    df_filtered_bump_log.loc[:, "Error Message"] = df_filtered_bump_log[
        "message"
    ].str.extract(r"- (.+)$")

    new_df = df_filtered_bump_log.reset_index().rename(columns={"index": "Time"})[
        ["Time", "ID", "Orientation", "Index", "Error Message"]
    ]

    # Extract measured force (ignore disabled actuators with zero force)
    new_df["MeasuredForce"] = (
        new_df["Error Message"].str.extract(r"\(([\-\d.]+)\)").astype(float)
    )
    new_df = new_df[new_df["MeasuredForce"] != 0]

    # Classify applied force direction
    new_df["AppliedForceDirection"] = new_df["Error Message"].apply(
        lambda x: "Positive" if "measured force plus" in x else "Negative"
    )

    # Calculate deviations
    upper_limit = expected_force_range
    lower_limit = -expected_force_range
    new_df["Deviation"] = 0.0

    for idx, row in new_df.iterrows():
        if row["AppliedForceDirection"] == "Positive":
            new_df.at[idx, "Deviation"] = float(row["MeasuredForce"]) - upper_limit
        elif row["AppliedForceDirection"] == "Negative":
            new_df.at[idx, "Deviation"] = float(row["MeasuredForce"]) - lower_limit

    # Classify deviation as overshoot or undershoot
    def classify_deviation(row):
        deviation = abs(row["MeasuredForce"]) - expected_force_range
        if deviation > 0:
            return "Overshoot"
        else:
            return "Undershoot"

    new_df["DeviationType"] = new_df.apply(classify_deviation, axis=1)

    cols = [
        "Time",
        "ID",
        "Orientation",
        "Index",
        "AppliedForceDirection",
        "MeasuredForce",
        "Deviation",
        "DeviationType",
    ]
    return new_df[cols]


### Plotting and Visualization of Bump Logs

In [12]:
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, Span, LabelSet, Legend, LegendItem, BoxAnnotation
from collections import defaultdict
from bokeh.layouts import gridplot

m1m3_actuator_id_index_table = {fa.actuator_id: fa.index for fa in FATable}

def get_m1m3_actuator_ids():
    """Get a list of the M1M3 actuator ids."""
    return list(m1m3_actuator_id_index_table.keys())


def get_xy_position(actuator_list=FATable):
    xpos = [actuator.x_position for actuator in actuator_list]
    ypos = [actuator.y_position for actuator in actuator_list]
    return xpos, ypos

def ActuatorsLayout_bokeh(df, actuator_list=FATable, width=400, height=400):
    """
    Plot the layout of M1M3 actuators and highlight failed actuators, using Bokeh.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns "ID" (actuator ID) and "Orientation" 
        for each failed actuator row.
    actuator_list : list or similar
        Typically FATable or any structure you use with get_xy_position(...)
        to get the x,y positions of each actuator.
    width, height : int
        Figure size in pixels.

    Returns
    -------
    p : bokeh.plotting.figure.Figure
        A Bokeh figure with all actuators + highlighted failures + legend.
    """
    orientation_colors = {"X": "blue", "Y": "red", "Z": "green"}
    combined_orientations_colors = {"XZ": "orange", "YZ": "black"}

    # Create the Bokeh figure
    p = figure(
        width=width,
        height=height,
        match_aspect=True,
        title="Failures Distribution",
        x_axis_label="X position (m)",
        y_axis_label="Y position (m)",
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )

    # 1) Plot all actuators faintly in the background
    ids = get_m1m3_actuator_ids()
    xpos, ypos = get_xy_position(actuator_list)

    source_all = ColumnDataSource(data=dict(
        x=xpos,
        y=ypos,
        id_str=[str(aid) for aid in ids],
    ))
    p.circle(
        x="x",
        y="y",
        source=source_all,
        size=14,
        alpha=0.05,
        color="blue",
        line_color="red",  # faint red outline
    )

    # Add small text labels for each actuator
    labels_all = LabelSet(
        x="x",
        y="y",
        text="id_str",
        text_font_size="6pt",
        text_color="blue",
        text_align="center",
        text_baseline="middle",
        x_offset=0,
        y_offset=0,
        source=source_all,
    )
    p.add_layout(labels_all)

    # 2) Identify which actuators have failed, by orientation
    actuator_orientations = defaultdict(set)
    for actuator_id, orientation in zip(df["ID"], df["Orientation"]):
        actuator_orientations[int(actuator_id)].add(orientation)

    # 3) For each failed actuator, draw a large hollow circle
    fail_x = []
    fail_y = []
    fail_colors = []

    for actuator_id, or_set in actuator_orientations.items():
        if actuator_id in m1m3_actuator_id_index_table:
            idx = m1m3_actuator_id_index_table[actuator_id]
            x_val = xpos[idx]
            y_val = ypos[idx]
            # Determine color from orientation
            if len(or_set) > 1:
                # e.g. orientations = {'X','Z'} => "XZ"
                combo = "".join(sorted(or_set))
                color = combined_orientations_colors.get(combo, "black")
            else:
                orientation = list(or_set)[0]  # single orientation
                color = orientation_colors.get(orientation, "black")

            fail_x.append(x_val)
            fail_y.append(y_val)
            fail_colors.append(color)

    source_fail = ColumnDataSource(data=dict(
        x=fail_x,
        y=fail_y,
        color=fail_colors,
    ))
    p.circle(
        x="x",
        y="y",
        source=source_fail,
        size=20,
        alpha=0.5,
        line_color="color",
        fill_color=None,
        line_width=2,
    )

    # 4) Build a Legend for single-orientation or combined-orientation
    used_orientations = set()
    used_combos = set()
    for or_set in actuator_orientations.values():
        if len(or_set) == 1:
            used_orientations |= or_set
        else:
            used_combos.add("".join(sorted(or_set)))

    legend_items = []

    # Single orientations
    for ori in sorted(used_orientations):
        color = orientation_colors.get(ori, "black")
        r = p.circle([], [], line_color=color, fill_color=None, size=10)
        legend_items.append(LegendItem(label=f"Orientation {ori}", renderers=[r]))

    # Combined orientations
    for combo in sorted(used_combos):
        color = combined_orientations_colors.get(combo, "black")
        r = p.circle([], [], line_color=color, fill_color=None, size=10)
        legend_items.append(LegendItem(label=f"Orientation {combo}", renderers=[r]))

    if legend_items:
        legend = Legend(items=legend_items, location="top_left")
        p.add_layout(legend, "right")

    return p


def plot_deviations_and_layout_bokeh(
    df_failures,
    actuator_list=FATable,
    expected_force=222,
    tolerance=5,
    width=600,
    height=300,
):
    """
    Creates a single Bokeh layout with:
      - Two plots stacked vertically (Positive, Negative) on the left
      - One ActuatorsLayout_bokeh figure on the right (with a square aspect),
        effectively spanning both rows.

    Mirrors the structure of your original Matplotlib 'plot_deviations_and_layout'.

    Parameters
    ----------
    df_failures : pd.DataFrame
        Must have columns: 'ID', 'DeviationType', 'AppliedForceDirection', 'MeasuredForce',
        'Orientation' (for ActuatorsLayout), etc.
    actuator_list : list or FATable
        The full M1M3 actuator table, used by ActuatorsLayout_bokeh.
    expected_force : float, default=222
        The nominal (expected) magnitude of the measured force.
    tolerance : float, default=5
        Allowed ± range from expected_force.
    width, height : int
        The size of each of the left plots (positive, negative). 
        The layout figure will be sized separately.

    Returns
    -------
    grid : bokeh.layouts.gridplot
        A 2×2 layout, where the right figure spans both rows.
    """

    # If df_failures is empty, replicate your original behavior
    if df_failures.empty:
        print("The DataFrame is empty. Nothing to plot.")
        return None

    # 1) Convert the ID to string for plotting
    df_failures["ID"] = df_failures["ID"].astype(str)
    sorted_ids = df_failures["ID"].sort_values().unique()
    id_to_position = {act_id: i for i, act_id in enumerate(sorted_ids)}
    df_failures["ID_Pos"] = df_failures["ID"].map(id_to_position)

    # 2) Separate positive and negative subsets
    df_positive = df_failures[df_failures["AppliedForceDirection"] == "Positive"]
    df_negative = df_failures[df_failures["AppliedForceDirection"] == "Negative"]

    # We’ll define color/marker by DeviationType
    # (You used 'o' for overshoot, 's' for undershoot in Matplotlib.)
    deviation_styles = {
        "Overshoot": {"color": "red", "marker": "circle"},
        "Undershoot": {"color": "green", "marker": "square"},
    }

    # -----------------------------------------------------------------
    # Create the top-left figure for "Positive Forces"
    # -----------------------------------------------------------------
    p_positive = figure(
        width=width,
        height=height,
        title="Measured Forces (Positive)",
        x_range=(-1, len(sorted_ids)),  # small margin
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p_positive.xaxis.axis_label = "FA ID"
    p_positive.yaxis.axis_label = "Measured Force (N)"

    for dev_type, group in df_positive.groupby("DeviationType"):
        style = deviation_styles.get(dev_type, {"color": "gray", "marker": "circle"})
        source_pos = ColumnDataSource(group)
        p_positive.scatter(
            x="ID_Pos",
            y="MeasuredForce",
            source=source_pos,
            color=style["color"],
            marker=style["marker"],
            alpha=0.7,
            size=8,
            legend_label=f"Positive - {dev_type}",
        )

    # Add a horizontal line for expected force
    span_expected = Span(
        location=expected_force,
        dimension="width",
        line_color="gray",
        line_dash="dashed",
        line_width=1,
    )
    p_positive.add_layout(span_expected)

    # Tolerance band
    band_pos = BoxAnnotation(
        bottom=expected_force - tolerance,
        top=expected_force + tolerance,
        fill_alpha=0.2,
        fill_color="gray",
    )
    p_positive.add_layout(band_pos)

    # X-axis label overrides: show FA IDs
    p_positive.xaxis.ticker = list(range(len(sorted_ids)))
    p_positive.xaxis.major_label_overrides = {
        i: sorted_ids[i] for i in range(len(sorted_ids))
    }

    p_positive.legend.location = "top_left"

    # -----------------------------------------------------------------
    # Create the bottom-left figure for "Negative Forces"
    # -----------------------------------------------------------------
    p_negative = figure(
        width=width,
        height=height,
        title="Measured Forces (Negative)",
        x_range=p_positive.x_range,  # share range
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    p_negative.xaxis.axis_label = "FA ID"
    p_negative.yaxis.axis_label = "Measured Force (N)"

    for dev_type, group in df_negative.groupby("DeviationType"):
        style = deviation_styles.get(dev_type, {"color": "gray", "marker": "circle"})
        source_neg = ColumnDataSource(group)
        p_negative.scatter(
            x="ID_Pos",
            y="MeasuredForce",
            source=source_neg,
            color=style["color"],
            marker=style["marker"],
            alpha=0.7,
            size=8,
            legend_label=f"Negative - {dev_type}",
        )

    # Horizontal line for negative expected force
    span_neg_expected = Span(
        location=-expected_force,
        dimension="width",
        line_color="gray",
        line_dash="dashed",
        line_width=1,
    )
    p_negative.add_layout(span_neg_expected)

    # Tolerance band for negative
    band_neg = BoxAnnotation(
        bottom=-expected_force - tolerance,
        top=-expected_force + tolerance,
        fill_alpha=0.2,
        fill_color="gray",
    )
    p_negative.add_layout(band_neg)

    p_negative.xaxis.ticker = list(range(len(sorted_ids)))
    p_negative.xaxis.major_label_overrides = {
        i: sorted_ids[i] for i in range(len(sorted_ids))
    }

    p_negative.legend.location = "top_left"

    # -----------------------------------------------------------------
    # Right subplot: the actuator layout (spanning both rows)
    # -----------------------------------------------------------------
    # Reuse our new Bokeh version of ActuatorsLayout
    p_layout = ActuatorsLayout_bokeh(df_failures, actuator_list, width=width, height=height*2)

    # -----------------------------------------------------------------
    # Combine them into a 2×2 grid, letting the layout figure fill the right side
    # -----------------------------------------------------------------
    # With gridplot, we can place `None` in the second row, right column 
    # so that p_layout effectively spans row 1 and row 2 on the right.
    from bokeh.layouts import gridplot

    grid = gridplot(
        [
            [p_positive, p_layout],
            [p_negative, None],
        ],
        merge_tools=False,  # or True if you want to share the same toolbar
    )

    return grid

def get_failed_actuator_ids(processed_bump_logs):
    try:
        failed_actuator_ids = processed_bump_logs["ID"].astype(int).unique()
        if len(failed_actuator_ids) == 0:
            raise ValueError("No failed actuators found in the processed bump logs.")
        return failed_actuator_ids
    except KeyError as e:
        print(f"KeyError: {e}. Ensure the DataFrame contains the 'ID' column.")
    except Exception as e:
        print(f"An error occurred: {e}")

## Detailed Force Actuator Bump Analysis

### Internal Helper Functions for Bump Analysis

In [13]:
def max_error(errors):
    return np.max([np.max(errors), np.max(errors * -1.0)])


def rms_times(t_start_str: str):
    """
    Return times where RMS calculation is performed, depending on a date threshold.
    """
    # This is the date of the IRQ change, which changed the bump test timings
    change_date = Time("2024-10-12T00:00:00", format="isot", scale="utc")
    t_start = Time(t_start_str, format="isot", scale="utc")

    if t_start < change_date:
        rms_t1 = 3.0
        rms_t2 = 4.0
        rms_t3 = 10.0
        rms_t4 = 11.0
    else:
        rms_t1 = 2.9
        rms_t2 = 3.9
        rms_t3 = 9.3
        rms_t4 = 10.3
    return [rms_t1, rms_t2, rms_t3, rms_t4]


def rms_error(times, errors, _rms_times):
    [rms_t1, rms_t2, rms_t3, rms_t4] = _rms_times
    error = 0.0
    num = 0
    for i, t in enumerate(times):
        if (t > rms_t1 and t < rms_t2) or (t > rms_t3 and t < rms_t4):
            num += 1
            error += errors[i] ** 2
    if num == 0:
        return np.nan
    else:
        return np.sqrt(error / num)

async def plot_bumps_and_errors_bokeh(
    client,
    bump_column,
    bt_result,
    force_column,
    follow_column,
    applied_column,
    p_s,
    actuator_id,
    start_time_str,
    end_time_str,
    BUMP_TEST_DURATION=14.0,
):
    """
    A Bokeh-based equivalent to plot_bumps_and_errors(...).

    It returns a list of 3 Bokeh figures:
      fig_row1, fig_row2, fig_row3
    corresponding to the three rows in your original Matplotlib code.

    Parameters
    ----------
    client : EfdClient
        An EFD client for data queries.
    bump_column : str
        Name of the column in bt_result that indicates TESTINGPOSITIVE state
        (e.g. "primaryTest3" or "secondaryTest5").
    bt_result : pd.DataFrame
        Bump test status data (like "past_days_data" or "bt_result").
    force_column : str
        e.g. "primaryCylinderForce3", the measured forces column name.
    follow_column : str
        e.g. "primaryCylinderFollowingError3", the following error column name.
    applied_column : str
        e.g. "zForces7" or "xForces2", the commanded (applied) force column.
    p_s : str
        "Primary", "Secondary", or any label for the subplot titles.
    actuator_id : int
        The force actuator ID for labeling.
    start_time_str, end_time_str : str
        ISO-format strings for the time range.
    BUMP_TEST_DURATION : float
        Duration (seconds) to query for each test. Default 14.0.

    Returns
    -------
    (fig_row1, fig_row2, fig_row3) : tuple of bokeh.plotting.figure.Figure
        Three Bokeh figures for row1, row2, row3 respectively.
    """
    # Filter for rows in bt_result that are in TESTINGPOSITIVE for this bump column.
    # e.g. if bump_column="primaryTest3", we want bt_result[bt_result["primaryTest3"] == BumpTest.TESTINGPOSITIVE]
    from lsst.ts.xml.enums.MTM1M3 import BumpTest
    df_pos = bt_result[bt_result[bump_column] == BumpTest.TESTINGPOSITIVE]
    df_pos = df_pos.sort_values("timestamp").reset_index(drop=True)

    measured_forces_times = []
    measured_forces_values = []
    following_error_values = []
    applied_forces_times = []
    applied_forces_values = []
    t_starts = []

    for idx in range(len(df_pos)):
        # Convert TAI (unix_tai) to astropy Time
        t_start_unix_tai = df_pos["timestamp"].iloc[idx] - 1.0  # minus 1.0 as in your code
        t_start = Time(t_start_unix_tai, format="unix_tai", scale="tai")
        t_end = t_start + TimeDelta(BUMP_TEST_DURATION, format="sec")

        t_starts.append(t_start.isot)

        # Query measured forces
        measured_forces = await client.select_time_series(
            "lsst.sal.MTM1M3.forceActuatorData",
            [force_column, follow_column, "timestamp"],
            t_start.utc,
            t_end.utc,
        )
        # Query applied forces
        applied_forces = await client.select_time_series(
            "lsst.sal.MTM1M3.appliedForces",
            [applied_column, "timestamp"],
            t_start.utc,
            t_end.utc,
        )

        if measured_forces.empty or applied_forces.empty:
            # If no data, store empty arrays
            measured_forces_times.append([])
            measured_forces_values.append([])
            following_error_values.append([])
            applied_forces_times.append([])
            applied_forces_values.append([])
            continue

        # Shift time zero based on the first measured_forces timestamp
        t0 = measured_forces["timestamp"].iloc[0]
        measured_forces["timestamp"] -= t0
        applied_forces["timestamp"] -= t0

        measured_forces_times.append(measured_forces["timestamp"].values)
        measured_forces_values.append(measured_forces[force_column].values)
        following_error_values.append(measured_forces[follow_column].values)
        applied_forces_times.append(applied_forces["timestamp"].values)
        applied_forces_values.append(applied_forces[applied_column].values)

    # We have N tests. We'll replicate your logic:
    #  Row 1: all tests, highlight the first's applied force, highlight the last's measured force
    #  Row 2: only the last test, plus RMS lines
    #  Row 3: following errors (RMS, max) vs test start time
    n_tests = len(t_starts)
    i_last = n_tests - 1

    # --------------------------------------------------------------------------
    # Row 1 figure: "Actuator {actuator_id} {p_s} forces vs time"
    # --------------------------------------------------------------------------
    fig_row1 = figure(
        width=500,
        height=300,
        x_range=(0, BUMP_TEST_DURATION),
        title=f"Actuator {actuator_id} {p_s} forces vs time",
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    fig_row1.xaxis.axis_label = "Time (sec.)"
    fig_row1.yaxis.axis_label = "Force (N)"

    if n_tests > 0:
        # Plot the first test’s applied force in gray
        fig_row1.line(
            applied_forces_times[0],
            applied_forces_values[0],
            line_color="gray",
            legend_label="First Test Applied",
        )

    # Plot all but the last test’s measured forces in gray
    for i in range(n_tests - 1):
        fig_row1.line(
            measured_forces_times[i],
            measured_forces_values[i],
            line_color="gray",
            line_width=1.0,
        )

    # Plot the last test’s measured forces in black (thicker)
    if i_last >= 0:
        fig_row1.line(
            measured_forces_times[i_last],
            measured_forces_values[i_last],
            line_color="black",
            line_width=2.0,
            legend_label="Latest Test Measured",
        )
        # Also the last test’s applied force in black dashed
        fig_row1.line(
            applied_forces_times[i_last],
            applied_forces_values[i_last],
            line_color="black",
            line_dash="dashed",
            line_width=1.5,
            legend_label="Latest Test Applied",
        )

    fig_row1.legend.location = "top_left"

    # --------------------------------------------------------------------------
    # Row 2 figure: "Last Test Only"
    # --------------------------------------------------------------------------
    last_title = f"Actuator {actuator_id} {p_s} forces {t_starts[i_last]}" if i_last >= 0 else ""
    fig_row2 = figure(
        width=500,
        height=300,
        x_range=(0, BUMP_TEST_DURATION),
        title=last_title,
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    fig_row2.xaxis.axis_label = "Time (sec.)"
    fig_row2.yaxis.axis_label = "Force (N)"

    if i_last >= 0:
        fig_row2.line(
            applied_forces_times[i_last],
            applied_forces_values[i_last],
            color="blue",
            legend_label="Commanded force",
        )
        fig_row2.line(
            measured_forces_times[i_last],
            measured_forces_values[i_last],
            color="red",
            legend_label="Measured force",
        )
        fig_row2.line(
            measured_forces_times[i_last],
            following_error_values[i_last],
            color="green",
            legend_label="Following error",
        )

        # Show vertical lines for RMS windows
        _rms_t = rms_times(t_starts[i_last])
        for marker in _rms_t:
            fig_row2.ray(
                x=marker,
                y=0,
                length=0,
                angle=np.pi/2,  # vertical
                line_dash="dashed",
                line_color="black",
            )

    fig_row2.legend.location = "best"

    # --------------------------------------------------------------------------
    # Row 3 figure: "Actuator {actuator_id} {p_s} following errors"
    # times vs RMS, times vs Max
    # --------------------------------------------------------------------------
    fig_row3 = figure(
        width=500,
        height=300,
        title=f"Actuator {actuator_id} {p_s} following errors",
        x_range=None,  # auto
        y_range=(0, 1000),  # similar to your code
        tools="pan,wheel_zoom,box_zoom,reset,save",
    )
    fig_row3.xaxis.major_label_orientation = np.pi/4  # rotate ~45 deg
    fig_row3.xaxis.axis_label = "Start Time (ISO)"
    fig_row3.yaxis.axis_label = "Errors (N)"
    # Bokeh has no built-in 'symlog' scale. We'll keep it linear or you can choose log:
    # fig_row3.y_scale = 'log'  # or keep linear

    times_x = []   # We'll store as strings (ISO) for the x-axis
    max_errors = []
    rms_errors_list = []

    for i in range(n_tests):
        times_x.append(t_starts[i])  # use the string as category or auto
        max_val = max_error(following_error_values[i]) if len(following_error_values[i]) else np.nan
        max_errors.append(max_val)

        _rms_times_val = rms_times(t_starts[i])
        rms_val = rms_error(
            measured_forces_times[i], 
            following_error_values[i], 
            _rms_times_val
        )
        rms_errors_list.append(rms_val)

    # We can treat times_x as a factor (categorical x-axis). Let's do that:
    fig_row3.x_range.factors = times_x

    # Plot RMS (blue x) and Max (green +)
    source_err = ColumnDataSource(data=dict(
        x=times_x,
        rms=rms_errors_list,
        mx=max_errors,
    ))
    fig_row3.scatter(
        x="x",
        y="rms",
        marker="x",
        size=8,
        color="blue",
        legend_label="RMS",
        source=source_err,
    )
    fig_row3.scatter(
        x="x",
        y="mx",
        marker="+",
        size=8,
        color="green",
        legend_label="Max",
        source=source_err,
    )
    fig_row3.legend.location = "best"
    
    fig_row3.yaxis.ticker = [0, 2, 4, 6, 8, 10, 50, 100, 500, 1000]

    # We could set y-range or do dynamic. 
    # If you want custom ticks [0, 2, 4, ...], 
    # you can do: fig_row3.yaxis.ticker = [0, 2, 4, 6, 8, 10, 50, 100, 500, 1000]

    return fig_row1, fig_row2, fig_row3


async def plot_actuator_error_bokeh(
    client,
    fa_id,
    bt_results,
    start_time_str,
    end_time_str,
    ndays=5,
    width=500,
    height=300,
):
    """
    Bokeh-based equivalent of plot_actuator_error(...),
    producing a 3×2 grid of Bokeh figures (or fewer if not a DAA actuator).

    Parameters
    ----------
    client : EfdClient
        The EFD client to query data.
    fa_id : int
        Force actuator ID to analyze.
    bt_results : pd.DataFrame
        DataFrame with bump test status for the relevant time window. 
        We'll filter rows for 'actuatorId' == fa_id.
    start_time_str, end_time_str : str
        ISO-format for time range.
    ndays : float
        How many days in the past to query the "past X days" data for the first triple.
    width, height : int
        Size of each Bokeh figure.

    Returns
    -------
    grid : bokeh.layouts.gridplot
        A 3×2 Bokeh layout (primary in column=0, secondary in column=1).
    """
    # 1) Identify which cylinder indices we need for this FA
    fa_data = force_actuator_from_id(fa_id)

    # 2) Filter out rows for this actuatorId
    bt_result_fa = bt_results[bt_results["actuatorId"] == fa_id]

    # 3) Query the "past X days" data
    past_days_data = await query_past_days_data(client, fa_id, end_time_str, days=ndays)

    # We'll produce 3 subplots for "Primary", 3 for "Secondary" (if DAA)
    figs_primary = []
    figs_secondary = []

    # --------------------------------------------------------------------------
    # Plot the "past X days" data for the primary
    # e.g. bump_column="primaryTest{fa_data.index}"
    # force_column="primaryCylinderForce{fa_data.index}"
    # follow_column="primaryCylinderFollowingError{fa_data.index}"
    # applied_column="zForces{fa_data.z_index}"  (typical for primary)
    bump_col_primary = f"primaryTest{fa_data.index}"
    force_col_primary = f"primaryCylinderForce{fa_data.index}"
    follow_col_primary = f"primaryCylinderFollowingError{fa_data.index}"
    applied_col_primary = f"zForces{fa_data.z_index}"  # from your original code

    figs_past_primary = await plot_bumps_and_errors_bokeh(
        client,
        bump_col_primary,
        past_days_data,
        force_col_primary,
        follow_col_primary,
        applied_col_primary,
        p_s="Primary",
        actuator_id=fa_id,
        start_time_str=start_time_str,
        end_time_str=end_time_str,
    )
    figs_primary.extend(figs_past_primary)

    # Next, plot the actual bumps from the selected range (bt_result_fa)
    figs_range_primary = await plot_bumps_and_errors_bokeh(
        client,
        bump_col_primary,
        bt_result_fa,
        force_col_primary,
        follow_col_primary,
        applied_col_primary,
        p_s="Primary",
        actuator_id=fa_id,
        start_time_str=start_time_str,
        end_time_str=end_time_str,
    )
    figs_primary.extend(figs_range_primary)

    # Now we have 6 figures, but actually we only want 3 total for "primary"? 
    # Wait, your original code calls `plot_bumps_and_errors(...)` once for "past 5 days" 
    # plus once for the actual lumps from the selected range => 2 calls each = 2 × 3 subplots = 6 subplots
    # So let's keep it consistent with the original: it merges them onto the same set of 3 subplots in Matplotlib. 
    # In Bokeh, let's simply do 6 separate figures, or just do the second call to overlay on the same. 
    # Merging them is more complex in Bokeh. We'll keep them separate for clarity.

    # We'll keep only the last 3 from the "actual lumps" call, or keep them all. 
    # For simplicity, let's keep them all:
    # figs_primary = [ row1past, row2past, row3past, row1range, row2range, row3range ]

    # --------------------------------------------------------------------------
    # Check if DAA (meaning we have a 'Secondary' cylinder)
    if fa_data.actuator_type.name == "DAA":
        # "secondaryTest{fa_data.s_index}", "secondaryCylinderForce{fa_data.s_index}", etc.
        bump_col_secondary = f"secondaryTest{fa_data.s_index}"
        force_col_secondary = f"secondaryCylinderForce{fa_data.s_index}"
        follow_col_secondary = f"secondaryCylinderFollowingError{fa_data.s_index}"

        # For the applied force column, we must determine xForces, yForces, or zForces 
        secondary_name = fa_data.orientation.name  # e.g. "X_PLUS", "Y_MINUS", "Z_PLUS", etc.
        if secondary_name in ["X_PLUS", "X_MINUS"]:
            applied_col_secondary = f"xForces{fa_data.x_index}"
        elif secondary_name in ["Y_PLUS", "Y_MINUS"]:
            applied_col_secondary = f"yForces{fa_data.y_index}"
        else:
            applied_col_secondary = f"zForces{fa_data.z_index}"

        figs_past_secondary = await plot_bumps_and_errors_bokeh(
            client,
            bump_col_secondary,
            past_days_data,
            force_col_secondary,
            follow_col_secondary,
            applied_col_secondary,
            p_s="Secondary",
            actuator_id=fa_id,
            start_time_str=start_time_str,
            end_time_str=end_time_str,
        )
        figs_secondary.extend(figs_past_secondary)

        figs_range_secondary = await plot_bumps_and_errors_bokeh(
            client,
            bump_col_secondary,
            bt_result_fa,
            force_col_secondary,
            follow_col_secondary,
            applied_col_secondary,
            p_s=secondary_name,  # e.g. "X_PLUS" or "Secondary"
            actuator_id=fa_id,
            start_time_str=start_time_str,
            end_time_str=end_time_str,
        )
        figs_secondary.extend(figs_range_secondary)

    else:
        # Non-DAA => fill with placeholders
        from bokeh.plotting import figure
        p1 = figure(width=width, height=height, title="Not a DAA actuator")
        p2 = figure(width=width, height=height, title="Not a DAA actuator")
        p3 = figure(width=width, height=height, title="Not a DAA actuator")
        figs_secondary = [p1, p2, p3]

    # We now have some number of figures in figs_primary (6 if we did two calls).
    # Typically you'd want to display the last 3 as the final result, or combine them. 
    # For demonstration, let's just pick the **first 3** from primary, first 3 from secondary 
    # to replicate the "3x2" final layout. You can adapt to your preference:
    figs_primary_display = figs_primary[:3]   # row1, row2, row3 for primary
    figs_secondary_display = figs_secondary[:3]  # row1, row2, row3 for secondary

    # Create a 3×2 grid
    # We'll place row i in [figs_primary_display[i], figs_secondary_display[i]]
    rows = []
    for i in range(3):
        row_figs = [figs_primary_display[i], figs_secondary_display[i]]
        rows.append(row_figs)

    grid = gridplot(rows, merge_tools=False)
    return grid

async def query_past_days_data(client, fa_id, end_time_str: str, days=5):
    """
    Query bump test status data for a specific actuator ID, from end_time_str going back 'days'.

    Parameters
    ----------
    client : EfdClient
        The EFD client to query data.
    fa_id : int
        The force actuator ID.
    end_time_str : str
        End time in ISO format, e.g. "2024-12-07T17:10:17.601".
    days : float
        How many days in the past to retrieve.

    Returns
    -------
    pd.DataFrame
        DataFrame with the bump test status results for the given time window.
    """
    end_time = Time(end_time_str, format="isot", scale="utc")
    start_time = end_time - TimeDelta(days, format="jd")

    bump_n_days_results = await client.select_time_series(
        "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatus",
        "*",
        start_time,
        end_time,
    )
    return bump_n_days_results

## Getting Past Executions

### Script Queue Logs
Here we create a summary of the last few executions of the `check_actuators.py` script. 

In [8]:
past_hours = convert_to_hours(past_time)

now_utc = datetime.utcnow()
end_dt = now_utc
start_dt = now_utc - timedelta(hours=past_hours)

start_str = start_dt.isoformat()
end_str = end_dt.isoformat()

script_logs = await query_script_queue_logs(
    start_str, end_str, client_name=notebook_client_name
)
script_logs_processed = filter_and_process_queue_logs(script_logs)
df_executions = extract_execution_details(script_logs_processed)

### Interactive Widget to Query and Plot Bump Test Failures

In this section, a set of **Jupyter Widgets** is provided to guide you through the bump test log analysis:

1. **`SalIndex` Dropdown:**
   Lets you select a specific script execution (identified by its `salIndex`) carried in the time range provided.
   Once selected, the notebook retrieves the associated execution details (start time, end time, duration, etc.).
   Note that executions shown here limited to the provided time range.

2. **`EFD Client` Dropdown:**
   Allows switching between available EFD clients (such as `usdf_efd` or `summit_efd`) if you need to query data from different environments.
   If doing analysis on real-time, the `summit_efd` might be the preferred client.

3. **“Get FA results” Button:**
   Initiates the actual query for bump test logs within the selected script execution’s time window.
   The results are then processed to identify any failed force actuators.

   Once the query is complete, the notebook displays a summary of the failed actuators, including the number of failed actuators and the list of their IDs.

   Optionally, one can check the "Plot measured forces" check button to plot the measured forces for all actuators.

4. **`FA ID` Dropdown:**
   Automatically populated with the list of failed actuator IDs found in the processed logs.
   Selecting an actuator ID triggers the retrieval of its forces, following errors, and other relevant data from the EFD.
   The notebook will then display diagnostic plots that focus on that specific actuator.

This widget-based workflow makes it easy to iterate through different script executions, check the resulting failed actuators, and visualize each one’s performance.

In [ ]:
# Display executions summary
if df_executions.empty:
    print("No executions found in the last {:.1f} days.".format(past_hours / 24.0))
else:
    print("Executions found in the last {:.1f} days.".format(past_hours / 24.0))
    display(HTML(df_executions.to_html(escape=False)))

Executions found in the last 36.0 days.


,scriptSalIndex,start_time,end_time,duration_minutes,FinalProcessStatus,FinalScriptStatus
0,103253,2024-12-11 18:05:55.151939154,2024-12-11 19:11:48.401677608,65.89,DONE,DONE
1,103246,2024-12-11 17:06:03.392565012,2024-12-11 17:06:31.659605265,0.47,DONE,DONE
2,103245,2024-12-11 17:01:13.748140812,2024-12-11 17:01:25.736535311,0.20,DONE,FAILED
3,103244,2024-12-11 16:55:13.111783981,2024-12-11 16:55:32.628724575,0.33,DONE,FAILED
4,102133,2024-12-10 21:41:42.211504936,2024-12-10 21:42:03.770392179,0.36,DONE,FAILED
5,102129,2024-12-10 21:22:12.441510439,2024-12-10 21:22:36.789692879,0.41,DONE,FAILED
6,102128,2024-12-10 20:11:07.505693674,2024-12-10 21:16:49.701204777,65.70,DONE,FAILED
7,101339,2024-12-09 22:15:07.831007957,2024-12-09 22:15:59.211351156,0.86,DONE,FAILED
8,101336,2024-12-09 21:31:16.331346750,2024-12-09 21:33:47.587690115,2.52,DONE,FAILED
9,101334,2024-12-09 21:16:45.981344461,2024-12-09 21:19:58.493230820,3.21,DONE,FAILED


In [10]:
%matplotlib inline

# -------------------------------------------------------------------
# Assumptions:
#   The following variables and functions are defined *elsewhere*
#   in the notebook or imported from external modules:
#   - df_executions: DataFrame listing script executions
#   - notebook_client_name: default EFD client name
#   - EfdClient, query_bump_logs, process_bump_logs:
#       helper functions to query and parse logs from the EFD
#   - get_failed_actuator_ids: extracts ID values from processed logs
#   - plot_actuator_error: plots detailed error data for a single FA ID
#   - plot_deviations_and_layout: a combined plot for measured forces
#     (positive/negative) and the layout distribution
# -------------------------------------------------------------------

################################################################################
# A global variable to remember the old callback function, so we can remove it
# when a new execution is selected. This ensures we don't double-trigger events.
################################################################################
previous_fa_id_callback = None

################################################################################
# 1) Define the interactive widgets
################################################################################

# Dropdown for picking a scriptSalIndex (execution) from df_executions
script_sal_index_selector = widgets.Dropdown(
    options=df_executions["scriptSalIndex"].unique(),
    value=df_executions["scriptSalIndex"].iloc[-1],  # default to the last one
    description="SalIndex:",
    disabled=False,
)

# Dropdown for choosing which EFD client to use (summit_efd, usdf_efd, etc.)
client_selector = widgets.Dropdown(
    options=["usdf_efd", "summit_efd"],
    value=notebook_client_name,
    description="EFD Client:",
    disabled=False,
)

# Checkbox to decide whether to plot combined measured forces and layout
show_forces_checkbox = widgets.Checkbox(
    value=False, description="Plot measured forces", disabled=False
)

# Button that triggers the main query workflow
query_button = widgets.Button(
    description="Get FA Results",
    disabled=False,
    button_style="info",
    tooltip="Click to query/process bump logs",
    icon="search",
)

# Dropdown for selecting a Force Actuator (FA) ID to plot detailed errors
fa_id_selector = widgets.Dropdown(
    options=[],
    description="FA ID:",
    disabled=False,
)

# Combine all widgets into a vertical layout, with the query button and
# checkbox side by side (HBox)
ui = widgets.VBox(
    [
        client_selector,
        script_sal_index_selector,
        widgets.HBox([query_button, show_forces_checkbox]),
        fa_id_selector,
    ]
)

################################################################################
# 2) Define the async callback for the "Get FA Results" button
################################################################################


async def on_query_button_clicked(b):
    """
    Handles the entire workflow when the user clicks the "Get FA Results" button.
    1. Clears previous output and redisplays the UI.
    2. Reads the selected execution from 'script_sal_index_selector'.
    3. Queries and processes bump logs from the EFD.
    4. Optionally plots measured forces if checkbox is checked.
    5. Displays summary of processed logs.
    6. Populates 'fa_id_selector' with new failed actuator IDs.
    7. Sets up a new callback for detailed FA ID plots.
    8. Automatically triggers the first ID in the list to plot immediately.
    """

    global previous_fa_id_callback

    # -----------------------------------------------------------------------
    # Clear the entire cell output each time the user clicks "Get FA Results"
    # This ensures a fresh start for logs and plots of the *new* execution.
    # -----------------------------------------------------------------------
    clear_output(wait=True)
    display(ui)

    # Gather user selections: scriptSalIndex + EFD client
    selected_script_sal_index = script_sal_index_selector.value
    execution_row = df_executions[
        df_executions["scriptSalIndex"] == selected_script_sal_index
    ].iloc[0]

    # Extract start/end times, durations, final statuses, etc.
    t_start = execution_row["start_time"]
    t_end = execution_row["end_time"]
    client_name = client_selector.value
    duration = execution_row["duration_minutes"]
    process_status = execution_row["FinalProcessStatus"]
    script_status = execution_row["FinalScriptStatus"]

    # Print basic info about the chosen execution
    print(
        f"Execution details:\n"
        f"    Script SalIndex: {selected_script_sal_index}\n"
        f"    Start Time: {t_start}\n"
        f"    End Time:   {t_end}\n"
        f"    Duration:   {duration} minutes\n"
        f"    Process Final Status: {process_status}\n"
        f"    Script Final Status:  {script_status}\n"
    )

    # Convert times to ISO format for astropy queries
    start_time_str = t_start.isoformat()
    end_time_str = t_end.isoformat()

    # If we had a previous callback from an older execution, unobserve it
    # to prevent multiple triggers.
    if previous_fa_id_callback is not None:
        fa_id_selector.unobserve(previous_fa_id_callback, names="value")
        previous_fa_id_callback = None

    # Create the EFD client to query logs
    client = EfdClient(client_name)

    print("Querying and processing bump log errors...")
    # Query the raw bump logs from the EFD
    bump_logs = await query_bump_logs(start_time_str, end_time_str, client_name)
    if bump_logs.empty:
        print("No bump test error logs found for the selected execution.")
        return

    # Process the logs (extract ID, DeviationType, etc.)
    processed_bump_logs = process_bump_logs(bump_logs)
    if processed_bump_logs.empty:
        print("No failed actuators found in the bump test logs.")
        return

    # If the user wants to see measured forces & layout, plot them now
    if show_forces_checkbox.value:
        print("\nMeasured positive/negative forces and failures distribution:")
        plot_deviations_and_layout(processed_bump_logs)
        plt.show()

    # Show the summary DataFrame of processed bump logs
    print("\nBump Test Error Summary:")
    display(processed_bump_logs)

    # Extract the IDs of failed actuators and populate the FA ID dropdown
    try:
        failed_actuator_ids = get_failed_actuator_ids(processed_bump_logs)
    except Exception as e:
        print(f"Error: {e}")
        return
    fa_id_selector.options = failed_actuator_ids

    client = EfdClient(client_name)
    # Query the bump test status data for more details
    bump_results = await client.select_time_series(
        "lsst.sal.MTM1M3.logevent_forceActuatorBumpTestStatus",
        "*",
        Time(start_time_str, format="isot", scale="utc"),
        Time(end_time_str, format="isot", scale="utc"),
    )
    if bump_results.empty:
        print("No bump test status data found for the selected execution.")
        return

    # Define a nested async function that plots error data for a specific FA ID
    async def plot_selected_actuator_errors(change):
        """
        Called when the user picks a new FA ID in 'fa_id_selector'.
        Plots detailed bump test error data for the chosen actuator.
        """
        fa_id = change["new"]
        # If 'fa_id' is None, ignore.
        if fa_id is None:
            return

        print(
            f"\nPlotting errors for Actuator ID: {fa_id}. "
            f"This may take a while, especially if 'ndays' > 5."
        )
        fig = plt.figure(figsize=(15, 20))
        await plot_actuator_error(
            fig, client, fa_id, bump_results, start_time_str, end_time_str
        )

    # This function is attached to the dropdown changes (value changes).
    def on_fa_id_change(change):
        asyncio.ensure_future(plot_selected_actuator_errors(change))

    # Observe the new callback
    fa_id_selector.observe(on_fa_id_change, names="value")
    # Store it so we can unobserve it next time
    previous_fa_id_callback = on_fa_id_change

    # Force a "real" change event to auto-plot the first ID.
    # 1) set to None so the widget sees a genuine change next
    # 2) set to the first item in the new list
    if len(fa_id_selector.options) > 0:
        fa_id_selector.value = None
        fa_id_selector.value = fa_id_selector.options[0]


# Attach the main callback to the "Get FA Results" button
query_button.on_click(lambda b: asyncio.ensure_future(on_query_button_clicked(b)))

# Finally, display the UI so the user can interact
display(ui)

NameError: name 'widgets' is not defined